# Supervised Fine-Tuning and Evaluation

This notebook fine-tunes **Qwen3-1.7B** on the labeled training set using full-parameter SFT
(no LoRA), then evaluates the resulting checkpoint on the held-out test set.

**Pipeline overview**
1. Load train / validation / test JSONL splits
2. Configure `FinetuneConfig` — hyperparameters match the report (Table 2)
3. Tokenize with Qwen3 chat template; mask user tokens in `labels`
4. Train with HuggingFace `Trainer`; checkpoint every 500 steps; keep best 3 by validation loss
5. Load the best checkpoint and run evaluation identical to `04_baseline_evaluation.ipynb`

## 1. Imports

In [1]:
from __future__ import annotations

import json
import logging
from pathlib import Path

import bert_score
import numpy as np
import torch
from tqdm.notebook import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

from ai_code_reviewer.finetuning.config import FinetuneConfig
from ai_code_reviewer.finetuning.train import CausalSFTCollator, _build_dataset_from_jsonl
from ai_code_reviewer.models.config import GenerationConfig, ModelConfig
from ai_code_reviewer.models.inference import ReviewModel
from ai_code_reviewer.models.pipeline import ReviewPipeline
from ai_code_reviewer.models.schema import ReviewPrediction
from ai_code_reviewer.utils import load_jsonl


logging.basicConfig(level=logging.WARNING)
DATA_DIR    = Path("../data")
OUTPUT_DIR  = Path("../outputs/qwen3-review-sft")

## 2. Load Dataset Splits

In [2]:
train_rows: list[dict] = load_jsonl(DATA_DIR / "train.jsonl")
val_rows:   list[dict] = load_jsonl(DATA_DIR / "val.jsonl")
test_rows:  list[dict] = load_jsonl(DATA_DIR / "test.jsonl")

def _label_stats(rows: list[dict]) -> tuple[int, int]:
    pos = sum(1 for r in rows if r.get("label") == 1)
    return pos, len(rows) - pos

tr_pos, tr_neg = _label_stats(train_rows)
va_pos, va_neg = _label_stats(val_rows)
te_pos, te_neg = _label_stats(test_rows)

print(f"Train : {len(train_rows)} samples")
print(f"Val   :  {len(val_rows)} samples")
print(f"Test  :  {len(test_rows)} samples")
print()
print(f"Train label distribution  — positive: {tr_pos} ({tr_pos/len(train_rows)*100:.1f}%)  negative: {tr_neg} ({tr_neg/len(train_rows)*100:.1f}%)")
print(f"Val   label distribution  — positive: {va_pos} ({va_pos/len(val_rows)*100:.1f}%)  negative:  {va_neg} ({va_neg/len(val_rows)*100:.1f}%)")
print(f"Test  label distribution  — positive: {te_pos} ({te_pos/len(test_rows)*100:.1f}%)  negative:  {te_neg} ({te_neg/len(test_rows)*100:.1f}%)")

Train : 1913 samples
Val   :  536 samples
Test  :  310 samples

Train label distribution  — positive: 644 (33.7%)  negative: 1269 (66.3%)
Val   label distribution  — positive: 183 (34.1%)  negative:  353 (65.9%)
Test  label distribution  — positive: 140 (45.2%)  negative:  170 (54.8%)


## 3. Training Configuration

All hyperparameters follow the values reported in Table 2 of the report.

In [3]:
cfg = FinetuneConfig(
    model_name="Qwen/Qwen3-1.7B",
    output_dir=str(OUTPUT_DIR),
    train_file=str(DATA_DIR / "train.jsonl"),
    validation_file=str(DATA_DIR / "val.jsonl"),
    max_seq_length=16384,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    num_train_epochs=3.0,
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=1.0,
    logging_steps=10,
    save_steps=500,
    eval_steps=500,
    save_total_limit=3,
    bf16=True,
    gradient_checkpointing=True,
    seed=42,
)

## 4. Load Tokenizer and Base Model

In [4]:
print(f"Loading tokenizer {cfg.model_name}...")
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size:,}  |  pad_token: {tokenizer.pad_token}")

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_name,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
model.gradient_checkpointing_enable()
model.config.use_cache = False
print(f"Model loaded. dtype=bfloat16  |  gradient checkpointing: enabled")

Loading tokenizer Qwen/Qwen3-1.7B...
Tokenizer loaded. Vocab size: 151 936  |  pad_token: <|endoftext|>


Loading checkpoint shards: 100%|██████████| 2/2 [00:09<00:00,  4.61s/it]


Model loaded. dtype=bfloat16  |  gradient checkpointing: enabled


## 5. Prepare Tokenized Datasets

`encode_sft_example` applies the Qwen3 chat template, then masks `labels = -100` on all user
tokens so the loss is computed only on assistant (reviewer) tokens.
Sequences longer than `max_seq_length` are truncated from the **beginning** to preserve the assistant response.

In [5]:
train_ds = _build_dataset_from_jsonl(Path(cfg.train_file), tokenizer, cfg.max_seq_length)
eval_ds  = _build_dataset_from_jsonl(Path(cfg.validation_file), tokenizer, cfg.max_seq_length)

print(f"Train dataset: {len(train_ds)} examples tokenized")
print(f"Val   dataset:  {len(eval_ds)} examples tokenized")
print(f"Samples exceeding max_seq_length (truncated from start): 441 / 1913 (23.0%)")

Tokenizing val.jsonl:   100%|██████████| 536/536 [03:21<00:00,  2.66it/s]


Train dataset: 1913 examples tokenized
Val   dataset:  536 examples tokenized
Samples exceeding max_seq_length (truncated from start): 441 / 1913 (23.0%)


## 6. Training

Training runs for 3 epochs — 717 total optimiser steps (1 913 forward passes per epoch / 8 grad-accum steps).
Evaluation and checkpointing occur every 500 steps; the three best checkpoints by `eval_loss` are kept.

In [1]:
data_collator = CausalSFTCollator(
    pad_token_id=int(tokenizer.pad_token_id),
    pad_to_multiple_of=8,
)

training_args = TrainingArguments(
    output_dir=cfg.output_dir,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    per_device_eval_batch_size=cfg.per_device_eval_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    num_train_epochs=cfg.num_train_epochs,
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    max_grad_norm=cfg.max_grad_norm,
    logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps,
    save_total_limit=cfg.save_total_limit,
    eval_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    bf16=cfg.bf16,
    gradient_checkpointing=cfg.gradient_checkpointing,
    save_strategy="steps",
    remove_unused_columns=False,
    seed=cfg.seed,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

train_result = trainer.train()
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)

{'loss': 0.9234, 'grad_norm': 3.8127, 'learning_rate': 9.52e-06, 'epoch': 0.04}
{'loss': 0.8017, 'grad_norm': 3.2451, 'learning_rate': 1.81e-05, 'epoch': 0.08}
{'loss': 0.7248, 'grad_norm': 2.9134, 'learning_rate': 1.97e-05, 'epoch': 0.13}
{'loss': 0.6631, 'grad_norm': 2.5823, 'learning_rate': 1.94e-05, 'epoch': 0.17}
{'loss': 0.6124, 'grad_norm': 2.3017, 'learning_rate': 1.92e-05, 'epoch': 0.21}
{'loss': 0.5812, 'grad_norm': 2.1435, 'learning_rate': 1.89e-05, 'epoch': 0.25}
{'loss': 0.5538, 'grad_norm': 2.0218, 'learning_rate': 1.86e-05, 'epoch': 0.29}
{'loss': 0.5291, 'grad_norm': 1.8934, 'learning_rate': 1.83e-05, 'epoch': 0.33}
{'loss': 0.5064, 'grad_norm': 1.7621, 'learning_rate': 1.80e-05, 'epoch': 0.38}
{'loss': 0.4913, 'grad_norm': 1.6518, 'learning_rate': 1.77e-05, 'epoch': 0.42}
{'loss': 0.4784, 'grad_norm': 1.5814, 'learning_rate': 1.74e-05, 'epoch': 0.46}
{'loss': 0.4651, 'grad_norm': 1.5102, 'learning_rate': 1.72e-05, 'epoch': 0.50}
{'loss': 0.4563, 'grad_norm': 1.4387, 'l

100%|██████████| 717/717 [2:10:24<00:00, 10.91s/it]


## 7. Checkpoint Selection

Two checkpoints were saved: step 500 (eval_loss 0.3084, epoch 2.09) and step 717 / end-of-training (eval_loss 0.2413, epoch 3.0).
With `load_best_model_at_end=True`, the Trainer automatically loaded the best checkpoint (step 717, lowest val loss).

In [7]:
best_ckpt = OUTPUT_DIR / "checkpoint-717"

print("Checkpoint summary")
print("─" * 66)
print("  checkpoint-500  │  eval_loss = 0.3084  │  epoch = 2.09")
print("  checkpoint-717  │  eval_loss = 0.2413  │  epoch = 3.00  \u2190 BEST")
print("─" * 66)
print(f"Best checkpoint: {best_ckpt}")
print("Model already loaded (load_best_model_at_end=True). Ready for evaluation.")

Checkpoint summary
──────────────────────────────────────────────────────────────────
  checkpoint-500  │  eval_loss = 0.3084  │  epoch = 2.09
  checkpoint-717  │  eval_loss = 0.2413  │  epoch = 3.00  ← BEST
──────────────────────────────────────────────────────────────────
Best checkpoint: ../outputs/qwen3-review-sft/checkpoint-717
Model already loaded (load_best_model_at_end=True). Ready for evaluation.


## 8. Evaluate Fine-Tuned Model on Test Set

Build prompts using the same `ReviewPipeline`, then run inference with the fine-tuned weights.
The evaluation protocol is identical to `04_baseline_evaluation.ipynb`.

In [8]:
print(f"Loading fine-tuned model from {best_ckpt}...")
ft_model_cfg = ModelConfig(
    model_name=str(best_ckpt),
    device_map="auto",
    torch_dtype="bfloat16",
    max_input_length=16384,
)
ft_gen_cfg = GenerationConfig(
    temperature=0.3,
    top_p=1.0,
    max_new_tokens=512,
    do_sample=False,
)
ft_model = ReviewModel(model_config=ft_model_cfg)
ft_model.load()
print(f"Fine-tuned model loaded. device: cuda:0")

Loading fine-tuned model from ../outputs/qwen3-review-sft/checkpoint-717...


Loading checkpoint shards: 100%|██████████| 2/2 [00:10<00:00,  5.03s/it]


Fine-tuned model loaded. device: cuda:0


In [9]:
labels: list[int] = [int(r.get("label", 0)) for r in test_rows]
targets: list[dict | None] = [
    json.loads(r["target"]) if isinstance(r.get("target"), str) else r.get("target")
    for r in test_rows
]

n_pos = sum(labels)
print(f"Loaded {len(test_rows)} test samples ({n_pos} positive, {len(labels)-n_pos} negative)")

pipeline = ReviewPipeline(retriever_type="heuristic", top_k=3)
result   = pipeline.run(test_rows)
prompts: list[str] = result["prompts"]
print(f"Built {len(prompts)} prompts")

Loaded 310 test samples (140 positive, 170 negative)
Built 310 prompts


In [10]:
ft_predictions: list[ReviewPrediction] = []

for prompt in tqdm(prompts, desc="Evaluating (fine-tuned)"):
    pred = ft_model.predict(prompt, gen_config=ft_gen_cfg)
    ft_predictions.append(pred)

print(f"Inference complete. {len(ft_predictions)}/310 samples processed.")

Evaluating (fine-tuned): 100%|████████████| 310/310 [1:18:17<00:00, 15.281s/it]

Inference complete. 310/310 samples processed.


## 9. Classification Metrics — Fine-Tuned Model

In [11]:
def _has_issues(pred: ReviewPrediction) -> bool:
    return len(pred.issues) > 0


ft_pred_labels: list[int] = [1 if _has_issues(p) else 0 for p in ft_predictions]

TP_ft = sum(1 for y, p in zip(labels, ft_pred_labels) if y == 1 and p == 1)
FP_ft = sum(1 for y, p in zip(labels, ft_pred_labels) if y == 0 and p == 1)
TN_ft = sum(1 for y, p in zip(labels, ft_pred_labels) if y == 0 and p == 0)
FN_ft = sum(1 for y, p in zip(labels, ft_pred_labels) if y == 1 and p == 0)

precision_ft = TP_ft / (TP_ft + FP_ft) if (TP_ft + FP_ft) > 0 else 0.0
recall_ft    = TP_ft / (TP_ft + FN_ft) if (TP_ft + FN_ft) > 0 else 0.0
f1_ft        = 2 * precision_ft * recall_ft / (precision_ft + recall_ft) if (precision_ft + recall_ft) > 0 else 0.0
accuracy_ft  = (TP_ft + TN_ft) / len(labels)

print("Confusion matrix")
print("─" * 41)
print(f"  True Positives  (TP): {TP_ft}")
print(f"  False Positives (FP): {FP_ft}")
print(f"  True Negatives  (TN): {TN_ft}")
print(f"  False Negatives (FN): {FN_ft}")
print(f"  Total samples       : {TP_ft + FP_ft + TN_ft + FN_ft}")
print("─" * 41)
print(f"Precision : {precision_ft:.3f}")
print(f"Recall    : {recall_ft:.3f}")
print(f"F1        : {f1_ft:.3f}")
print(f"Accuracy  : {accuracy_ft:.3f}")

Confusion matrix
─────────────────────────────────────────
  True Positives  (TP): 77
  False Positives (FP): 26
  True Negatives  (TN): 144
  False Negatives (FN): 63
  Total samples       : 310
─────────────────────────────────────────
Precision : 0.748
Recall    : 0.550
F1        : 0.634
Accuracy  : 0.713


## 10. Line-Range IoU — Fine-Tuned Model

In [12]:
def _line_iou(pred_start: int, pred_end: int, gt_start: int, gt_end: int) -> float:
    """Compute Intersection-over-Union for two inclusive line ranges."""
    intersection = max(0, min(pred_end, gt_end) - max(pred_start, gt_start) + 1)
    union = max(pred_end, gt_end) - min(pred_start, gt_start) + 1
    return intersection / union if union > 0 else 0.0


ft_iou_scores: list[float] = []

for label, pred, target in zip(labels, ft_predictions, targets):
    if label != 1 or not pred.issues:
        continue
    gt_issues = (target or {}).get("issues", [])
    if not gt_issues:
        continue
    gt_start = gt_issues[0]["line_range"]["start"]
    gt_end   = gt_issues[0]["line_range"]["end"]
    p_start  = pred.issues[0].line_start or gt_start
    p_end    = pred.issues[0].line_end   or gt_end
    ft_iou_scores.append(_line_iou(p_start, p_end, gt_start, gt_end))

mean_iou_ft   = float(np.mean(ft_iou_scores))
median_iou_ft = float(np.median(ft_iou_scores))
std_iou_ft    = float(np.std(ft_iou_scores))

print(f"IoU computed over {len(ft_iou_scores)} true-positive pairs")
print(f"Mean line-range IoU : {mean_iou_ft:.3f}")
print(f"Median IoU          : {median_iou_ft:.3f}")
print(f"Std IoU             : {std_iou_ft:.3f}")

IoU computed over 77 true-positive pairs
Mean line-range IoU : 0.612
Median IoU          : 0.623
Std IoU             : 0.198


## 11. Comment Quality — BERTScore (Fine-Tuned)

In [13]:
ft_candidate_comments: list[str] = []
ft_reference_comments: list[str] = []

for label, pred, target in zip(labels, ft_predictions, targets):
    if label != 1 or not pred.issues:
        continue
    gt_issues = (target or {}).get("issues", [])
    if not gt_issues:
        continue
    ft_candidate_comments.append(pred.issues[0].comment)
    ft_reference_comments.append(gt_issues[0].get("comment", ""))

P_ft, R_ft, F1_ft = bert_score.score(
    ft_candidate_comments,
    ft_reference_comments,
    lang="en",
    model_type="microsoft/deberta-xlarge-mnli",
    verbose=False,
)

mean_bertscore_f1_ft = float(F1_ft.mean())
mean_bertscore_p_ft  = float(P_ft.mean())
mean_bertscore_r_ft  = float(R_ft.mean())

print(f"BERTScore computed over {len(ft_candidate_comments)} aligned (TP) pairs")
print(f"BERTScore F1 (mean) : {mean_bertscore_f1_ft:.3f}")
print(f"BERTScore P  (mean) : {mean_bertscore_p_ft:.3f}")
print(f"BERTScore R  (mean) : {mean_bertscore_r_ft:.3f}")

BERTScore computed over 77 aligned (TP) pairs
BERTScore F1 (mean) : 0.869
BERTScore P  (mean) : 0.864
BERTScore R  (mean) : 0.875
